# 02 — Generate Dataset CNN dari YOLOv12

Tahap **[3]–[5]** pipeline: YOLOv12 terlatih dijalankan pada seluruh foto
(train/valid/test), tiap deteksi dicocokkan ke anotasi ground truth dengan
IoU ≥ 0.5, lalu wilayah prediksi yang cocok di-crop dan diberi label dari
ground truth. Hasilnya mengisi `data/crops/` — **satu-satunya sumber data**
untuk melatih dan menguji ketiga CNN.

**Prasyarat:** YOLOv12 sudah dilatih dan bobot terbaiknya disalin ke
`results/models/yolov12_best.pt` (notebook 03 / `scripts.train --model yolo`).

Yang perlu diperhatikan saat membaca hasil:

| Istilah | Arti | Konsekuensi |
| --- | --- | --- |
| **TP (matched)** | deteksi cocok dengan anotasi | menjadi crop dataset CNN |
| **FN** | biji ada di anotasi tetapi tidak terdeteksi | **hilang** dari evaluasi CNN |
| **FP** | deteksi tanpa pasangan anotasi | dibuang (label tidak diketahui) |

Ekuivalen CLI: `python -m scripts.generate_cnn_dataset`

In [ ]:
# Bootstrap: tambahkan root proyek ke sys.path agar modul src/ & configs/ dapat diimpor.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
import random

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from configs.base_config import (
    ANNOTATED_DATA_DIR,
    CLASS_NAMES,
    CROPS_DIR,
    IOU_MATCH_THRESHOLD,
    MODELS_DIR,
    RANDOM_SEED,
)
from src.data.crop_from_yolo import generate_all_splits, load_yolo_model, predict_boxes
from src.evaluation.matching import match_predictions_to_ground_truth
from src.utils.io import parse_yolo_label_file
from src.utils.seed import set_seed

set_seed(RANDOM_SEED)

## Muat YOLOv12 terlatih

In [ ]:
YOLO_WEIGHTS = MODELS_DIR / 'yolov12_best.pt'
print(YOLO_WEIGHTS, '->', 'ADA' if YOLO_WEIGHTS.exists() else 'BELUM ADA — latih YOLOv12 dulu (notebook 03)')

yolo_model = load_yolo_model(YOLO_WEIGHTS)

## Generate crop untuk train / valid / test

Model dimuat sekali lalu dipakai ulang antar-split. Statistik matching ditulis
ke `data/crops/matching_stats.json`.

In [ ]:
results = generate_all_splits(
    yolo_model_path=YOLO_WEIGHTS,
    dataset_root=ANNOTATED_DATA_DIR,
    output_root=CROPS_DIR,
    class_names=CLASS_NAMES,
    iou_threshold=IOU_MATCH_THRESHOLD,
)
print('statistik:', results['stats_path'])

## Statistik matching per split

In [ ]:
rows = {split: stats['summary'] for split, stats in results['splits'].items()}
rows['TOTAL'] = results['total']
stats_df = pd.DataFrame(rows).T[
    ['num_images', 'num_ground_truths', 'num_matched', 'num_false_negatives',
     'num_false_positives', 'num_crops_saved', 'yolo_recall', 'yolo_precision']
]
display(stats_df.round(4))

total = results['total']
print(
    f"\n{total['num_false_negatives']} dari {total['num_ground_truths']} biji ground truth "
    f"tidak terdeteksi YOLOv12 → tidak pernah dievaluasi CNN.\n"
    f"Angka ini WAJIB dilaporkan bersama metrik CNN (lihat notebook 07)."
)

In [ ]:
if results['splits']:
    plot_df = stats_df.drop(index='TOTAL', errors='ignore')
    ax = plot_df[['num_matched', 'num_false_positives', 'num_false_negatives']].plot(
        kind='bar', figsize=(9, 5), color=['tab:green', 'tab:orange', 'tab:red'],
    )
    ax.set_ylabel('jumlah biji'); ax.set_xlabel('split')
    ax.set_title(f'Hasil Matching YOLOv12 ↔ Ground Truth (IoU ≥ {IOU_MATCH_THRESHOLD})')
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Visualisasi sampel: deteksi vs anotasi

- **Hijau (garis penuh)** — deteksi YOLOv12 yang *matched* → jadi crop dataset CNN
- **Kuning (garis penuh)** — deteksi *false positive* → dibuang
- **Putih (garis putus-putus)** — anotasi ground truth

In [ ]:
SPLIT = 'test'
images_dir = ANNOTATED_DATA_DIR / SPLIT / 'images'
labels_dir = ANNOTATED_DATA_DIR / SPLIT / 'labels'

image_paths = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
random.seed(RANDOM_SEED)
samples = random.sample(image_paths, k=min(2, len(image_paths)))

for image_path in samples:
    image = Image.open(image_path).convert('RGB')
    detections = predict_boxes(yolo_model, image_path)
    ground_truths = parse_yolo_label_file(labels_dir / f'{image_path.stem}.txt', *image.size)
    matching = match_predictions_to_ground_truth(detections, ground_truths, IOU_MATCH_THRESHOLD)

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(image); ax.axis('off')

    for gt in ground_truths:  # anotasi ground truth
        x1, y1, x2, y2 = gt['bbox']
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor='white', linewidth=1.2, linestyle='--'))

    for pair in matching['matched_pairs']:  # deteksi matched -> crop CNN
        x1, y1, x2, y2 = pair['pred_bbox']
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor='lime', linewidth=2))
        ax.text(x1, max(y1 - 4, 0), f"{CLASS_NAMES[pair['gt_class_id']]} iou={pair['iou']:.2f}",
                fontsize=7, color='black',
                bbox={'facecolor': 'lime', 'alpha': 0.75, 'pad': 1, 'edgecolor': 'none'})

    for fp in matching['false_positives']:  # deteksi tanpa pasangan GT
        x1, y1, x2, y2 = fp['bbox']
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor='yellow', linewidth=2))

    ax.set_title(
        f"{image_path.name} — matched={len(matching['matched_pairs'])} "
        f"FP={len(matching['false_positives'])} FN={len(matching['false_negatives'])}"
    )
    plt.tight_layout(); plt.show()

## Verifikasi visual crop hasil

Grid crop per kelas — pastikan tiap crop benar-benar berisi satu biji utuh dan
labelnya masuk akal sebelum melanjutkan ke pelatihan CNN.

In [ ]:
N_SHOW = 8

for class_name in CLASS_NAMES:
    crop_paths = sorted((CROPS_DIR / SPLIT / class_name).glob('*.jpg'))[:N_SHOW]
    if not crop_paths:
        print(f'Belum ada crop untuk kelas {class_name} di split {SPLIT}.')
        continue

    fig, axes = plt.subplots(1, len(crop_paths), figsize=(2 * len(crop_paths), 2.4))
    for ax, path in zip([axes] if len(crop_paths) == 1 else axes.ravel(), crop_paths):
        ax.imshow(Image.open(path)); ax.axis('off'); ax.set_title(path.stem[-12:], fontsize=6)
    fig.suptitle(f'Crop kelas "{class_name}" (split {SPLIT}) — sumber: prediksi YOLOv12')
    plt.tight_layout(); plt.show()

for split in ['train', 'valid', 'test']:
    counts = {c: len(list((CROPS_DIR / split / c).glob('*.jpg'))) for c in CLASS_NAMES
              if (CROPS_DIR / split / c).is_dir()}
    if counts:
        print(f'{split}: {counts} (total {sum(counts.values())})')